# 1. Baseline Model Training (XG Boost)

In [5]:
# Imports

import pandas as pd
import numpy as np
import time
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

In [6]:
# Path Configuration 

DATA_DIR   = './Dataset_Cleaned/'
OUTPUT_DIR = './outputs/baseline/'
SEED       = 42
CV_FOLDS   = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)

print("Configuration loaded.")
print(f"  Data directory   : {DATA_DIR}")
print(f"  Output directory : {OUTPUT_DIR}")
print(f"  Random seed      : {SEED}")

Configuration loaded.
  Data directory   : ./Dataset_Cleaned/
  Output directory : ./outputs/baseline/
  Random seed      : 42


In [ ]:
# Load pre-split dataset 

print("Loading pre-split datasets...")

X_train = pd.read_csv(DATA_DIR + 'X_train_smote.csv')
X_test  = pd.read_csv(DATA_DIR + 'X_test.csv')
y_train = pd.read_csv(DATA_DIR + 'y_train_smote.csv').squeeze()
y_test  = pd.read_csv(DATA_DIR + 'y_test.csv').squeeze()

feature_names = X_train.columns.tolist()

print(f"  X_train shape : {X_train.shape}")
print(f"  X_test  shape : {X_test.shape}")
print(f"  y_train classes : {sorted(y_train.unique())}")
print(f"  y_test  classes : {sorted(y_test.unique())}")
print(f"\n  Training class distribution:")
for cls, cnt in y_train.value_counts().items():
    print(f"    {cls:<40} {cnt:>8,}  ({cnt/len(y_train)*100:.2f}%)")

Loading pre-split datasets...
  X_train shape : (3354374, 45)
  X_test  shape : (504473, 45)
  y_train classes : [np.int64(0), np.int64(1)]
  y_test  classes : [np.int64(0), np.int64(1)]

  Training class distribution:
    0                                        1,677,187  (50.00%)
    1                                        1,677,187  (50.00%)


In [18]:
# Encode Labels

print("Setting up class labels...")

class_names = ['BENIGN', 'ATTACK']
n_classes   = 2

y_train_enc = y_train.values
y_test_enc  = y_test.values

print(f"  Number of classes : {n_classes}")
print(f"  Classes           : {class_names}")
print(f"  y_train dtype     : {y_train_enc.dtype}")
print(f"  y_test  dtype     : {y_test_enc.dtype}")

# Save class names for use in metaheuristic notebooks
joblib.dump(class_names, OUTPUT_DIR + 'label_encoder.pkl')
print(f"\n  Class names saved → {OUTPUT_DIR}label_encoder.pkl")

Setting up class labels...
  Number of classes : 2
  Classes           : ['BENIGN', 'ATTACK']
  y_train dtype     : int64
  y_test  dtype     : int64

  Class names saved → ./outputs/baseline/label_encoder.pkl


In [19]:
# Load and verify scaling
# The cleaning notebook already scaled the data using MinMaxScaler.
# We load it here for reuse in metaheuristic notebooks.
# We do NOT re-scale X_train or X_test — they are already scaled.

print("Loading MinMaxScaler from cleaning notebook...")

scaler = joblib.load(DATA_DIR + 'minmax_scaler.pkl')

print("  Scaler loaded successfully.")
print(f"  Feature range check (X_train):")
print(f"    Min  : {X_train.values.min():.4f}")
print(f"    Max  : {X_train.values.max():.4f}")
print(f"    Mean : {X_train.values.mean():.4f}")
print("\n  Data is already scaled — no re-scaling applied.")

# Save a copy to the output folder for metaheuristic notebooks
joblib.dump(scaler, OUTPUT_DIR + 'scaler.pkl')
print(f"  Scaler copy saved → {OUTPUT_DIR}scaler.pkl")

Loading MinMaxScaler from cleaning notebook...
  Scaler loaded successfully.
  Feature range check (X_train):
    Min  : 0.0000
    Max  : 1.0000
    Mean : 0.1027

  Data is already scaled — no re-scaling applied.
  Scaler copy saved → ./outputs/baseline/scaler.pkl


In [20]:
# Even after SMOTE, minor residual imbalance may exist.
# Sample weights ensure the model penalises misclassification
# of rarer classes more heavily during training.

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_enc)

print("Sample weights computed.")
print(f"  Min weight : {sample_weights.min():.6f}")
print(f"  Max weight : {sample_weights.max():.6f}")
print(f"  Mean weight: {sample_weights.mean():.6f}")

Sample weights computed.
  Min weight : 1.000000
  Max weight : 1.000000
  Mean weight: 1.000000


In [21]:
# =============================================================================
# CELL 6 — Define and Train XGBoost Baseline (FIXED)
# =============================================================================
# Labels are binary (0/1) so objective must be binary:logistic,
# not multi:softprob. Remove num_class parameter entirely.

print("=" * 60)
print("  Training XGBoost Baseline Model")
print("=" * 60)

xgb_baseline = XGBClassifier(
    # Tree parameters
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.1,
    # Sampling regularisation
    subsample        = 0.8,
    colsample_bytree = 0.8,
    # Binary classification
    objective        = 'binary:logistic',
    eval_metric      = 'logloss',
    # Performance
    tree_method      = 'hist',
    n_jobs           = -1,
    random_state     = SEED,
    verbosity        = 0
)

print("\nModel parameters:")
for k, v in xgb_baseline.get_params().items():
    print(f"  {k:<25} : {v}")

print("\nFitting model...")
train_start = time.time()

xgb_baseline.fit(
    X_train, y_train_enc,
    sample_weight = sample_weights,
    verbose       = False
)

train_time = time.time() - train_start
print(f"\n  Training complete in {train_time:.2f}s")


  Training XGBoost Baseline Model

Model parameters:
  objective                 : binary:logistic
  base_score                : None
  booster                   : None
  callbacks                 : None
  colsample_bylevel         : None
  colsample_bynode          : None
  colsample_bytree          : 0.8
  device                    : None
  early_stopping_rounds     : None
  enable_categorical        : False
  eval_metric               : logloss
  feature_types             : None
  feature_weights           : None
  gamma                     : None
  grow_policy               : None
  importance_type           : None
  interaction_constraints   : None
  learning_rate             : 0.1
  max_bin                   : None
  max_cat_threshold         : None
  max_cat_to_onehot         : None
  max_delta_step            : None
  max_depth                 : 6
  max_leaves                : None
  min_child_weight          : None
  missing                   : nan
  monotone_constraints      

In [22]:
# =============================================================================
# CELL 7 — Evaluate on Test Set (FIXED)
# =============================================================================

print("=" * 60)
print("  Test Set Evaluation")
print("=" * 60)

y_pred      = xgb_baseline.predict(X_test)
y_pred_prob = xgb_baseline.predict_proba(X_test)

# ── Core metrics ──────────────────────────────────────────────────────────────
acc       = accuracy_score(y_test_enc, y_pred)
f1_w      = f1_score(y_test_enc, y_pred, average='weighted',  zero_division=0)
f1_macro  = f1_score(y_test_enc, y_pred, average='macro',     zero_division=0)
precision = precision_score(y_test_enc, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test_enc, y_pred, average='weighted',    zero_division=0)

# ── ROC-AUC (binary — use ATTACK class probability column) ───────────────────
try:
    roc_auc = roc_auc_score(y_test_enc, y_pred_prob[:, 1])
except Exception as e:
    roc_auc = float('nan')
    print(f"  ROC-AUC failed: {e}")

# ── False Positive Rate ───────────────────────────────────────────────────────
cm = confusion_matrix(y_test_enc, y_pred)
TN, FP, FN, TP = cm.ravel()
fpr_macro = FP / (FP + TN)

print(f"\n  Accuracy            : {acc:.6f}  ({acc*100:.4f}%)")
print(f"  Weighted F1         : {f1_w:.6f}")
print(f"  Macro F1            : {f1_macro:.6f}")
print(f"  Weighted Precision  : {precision:.6f}")
print(f"  Weighted Recall     : {recall:.6f}")
print(f"  ROC-AUC             : {roc_auc:.6f}")
print(f"  False Positive Rate : {fpr_macro:.6f}  ({fpr_macro*100:.4f}%)")
print(f"  Training Time (s)   : {train_time:.2f}")
print(f"  Features Used       : {len(feature_names)}")

print(f"\n  Per-Class Report:")
print(classification_report(y_test_enc, y_pred, target_names=class_names, zero_division=0))

# ── Save metrics CSV ──────────────────────────────────────────────────────────
baseline_metrics = pd.DataFrame([{
    'Model'           : 'XGBoost Baseline',
    'Accuracy'        : round(acc, 6),
    'Weighted_F1'     : round(f1_w, 6),
    'Macro_F1'        : round(f1_macro, 6),
    'Precision'       : round(precision, 6),
    'Recall'          : round(recall, 6),
    'ROC_AUC'         : round(roc_auc, 6),
    'FPR'             : round(fpr_macro, 6),
    'Features_Used'   : len(feature_names),
    'Training_Time_s' : round(train_time, 2)
}])
baseline_metrics.to_csv(OUTPUT_DIR + 'baseline_metrics.csv', index=False)
print(f"\n  Metrics saved → {OUTPUT_DIR}baseline_metrics.csv")


  Test Set Evaluation

  Accuracy            : 0.999005  (99.9005%)
  Weighted F1         : 0.999006
  Macro F1            : 0.998231
  Weighted Precision  : 0.999010
  Weighted Recall     : 0.999005
  ROC-AUC             : 0.999976
  False Positive Rate : 0.001140  (0.1140%)
  Training Time (s)   : 9.28
  Features Used       : 45

  Per-Class Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    419297
      ATTACK       0.99      1.00      1.00     85176

    accuracy                           1.00    504473
   macro avg       1.00      1.00      1.00    504473
weighted avg       1.00      1.00      1.00    504473


  Metrics saved → ./outputs/baseline/baseline_metrics.csv
